# 01 · Prompting baselines & RQ1: training-token budget (P2)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Syndr0/compute-efficient-genqa/blob/main/notebooks/01_baselines_and_rq1_training_budget.ipynb)

1. **Zero-shot** and **3-shot** prompting of Qwen3-0.6B on all five eval sets.
2. **LoRA SFT** on the SQuAD + NewsQA + TriviaQA mixture at budgets of 0.25M, 1M and 4M
   training tokens (configurable in `configs/default.yaml → rq1`), each evaluated on all eval sets.

Logged per run: training loss, dev loss at 7 points, tokens/s, GPU time, estimated cost and FLOPs,
peak memory. Runtime on a T4: ~1 h.

In [ ]:
#@title Setup: clone the repo, install, choose where outputs go
#@markdown Outputs (data, adapters, predictions, metrics) persist on Google Drive so a
#@markdown disconnected runtime can resume: every stage skips work that is already saved.
USE_DRIVE = True  #@param {type:"boolean"}
SMOKE_TEST = False  #@param {type:"boolean"}
GPU_PROFILE = "auto"  #@param ["auto", "t4", "a100"]
import os, sys
REPO_DIR = "/content/compute-efficient-genqa"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/Syndr0/compute-efficient-genqa.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/cegqa_outputs"
else:
    OUT = "/content/cegqa_outputs"
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
if GPU_PROFILE == "auto":
    GPU_PROFILE = "a100" if any(k in GPU for k in ("A100", "H100")) else "t4"
CONFIG_FILES = ["configs/default.yaml"]
if GPU_PROFILE == "a100":
    CONFIG_FILES.append("configs/a100.yaml")
    OUT += "_a100"  # different budgets/sizes: keep results of the two profiles apart
if SMOKE_TEST:
    CONFIG_FILES.append("configs/smoke.yaml")
    OUT += "_smoke"
os.environ["CEGQA_OUTPUT_DIR"] = OUT
CONFIG = " ".join(f"--config {f}" for f in CONFIG_FILES)
print(f"GPU: {GPU} | profile: {GPU_PROFILE} | outputs -> {OUT}")
print("configs:", CONFIG_FILES)

In [ ]:
!python -m cegqa.cli prepare $CONFIG

## Prompting baselines

In [ ]:
!python -m cegqa.cli baselines $CONFIG

## LoRA fine-tuning at each token budget

In [ ]:
!python -m cegqa.cli rq1 $CONFIG

## Results

In [ ]:
from cegqa.utils import load_config
cfg = load_config(CONFIG_FILES)

In [ ]:
import json, pandas as pd
from pathlib import Path
from cegqa.data import eval_sources
from cegqa.train_lora import run_name
runs = Path(cfg["output_dir"]) / "runs"
rows = []
for mode in cfg["baselines"]["modes"]:
    m = json.load(open(runs / f"baseline_{mode}" / "metrics.json"))
    rows.append({"system": mode, **{f"{s} F1": round(m[s]["f1"], 1) for s in eval_sources(cfg)}})
for b in cfg["rq1"]["token_budgets"]:
    d = runs / run_name(cfg["rq1"]["sources"], int(b), cfg["rq1"]["seeds"][0])
    m, s = json.load(open(d / "metrics.json")), json.load(open(d / "train_summary.json"))
    rows.append({"system": f"LoRA {b:,} tok", "GPU-min": round(s["train_time_s"] / 60, 1),
                 "dev loss": round(s["final_dev_loss"], 3), **{f"{x} F1": round(m[x]["f1"], 1) for x in eval_sources(cfg)}})
pd.DataFrame(rows)

In [ ]:
!python -m cegqa.cli report $CONFIG
from IPython.display import Image, display
for f in ["rq1_budget_curves.png", "rq1_dev_loss.png", "rq1_f1_vs_gpu_minutes.png"]:
    display(Image(f"{cfg['output_dir']}/figures/{f}"))